# Lab MLOps — Fine-tuning d'un Vision Transformer (ViT) sur Hugging Face

**Objectif :** entraîner (fine-tuner) un modèle de classification d'images sur un dataset Hugging Face, l'évaluer, puis le **publier sur le Hugging Face Hub**. Ensuite (hors notebook) vous créerez un **Space Gradio** qui utilise ce modèle.

**Dataset :** [`beans`](https://huggingface.co/datasets/beans) — feuilles de haricot, 3 classes (`angular_leaf_spot`, `bean_rust`, `healthy`). Petit et rapide, idéal pour un atelier.

**Modèle de base :** `google/vit-base-patch16-224-in21k` (Vision Transformer pré-entraîné).

**Environnement :** Google Colab avec **GPU** — menu `Runtime > Change runtime type > T4 GPU`.

> ⏱️ Le fine-tuning prend ~3 à 6 minutes sur un GPU T4.


## 1. Installation des dépendances

PyTorch et torchvision sont déjà installés sur Colab. On ajoute la stack Hugging Face.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate
print("Dépendances installées.")

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  Pas de GPU détecté. Allez dans Runtime > Change runtime type > T4 GPU, puis relancez.")

## 2. Connexion au Hugging Face Hub

Il faut un compte Hugging Face (gratuit) et un **token d'accès en écriture** :
`huggingface.co > Settings > Access Tokens > New token` (rôle **Write**).

Exécutez la cellule ci-dessous et collez votre token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Chargement et exploration du dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("beans")
print(dataset)

# Le dataset a déjà des splits train / validation / test.
example = dataset["train"][0]
print("Colonnes :", dataset["train"].column_names)
print("Label du 1er exemple :", example["labels"])
example["image"]  # affiche l'image (PIL)

In [ ]:
# Récupération des noms de classes et des dictionnaires id<->label
labels = dataset["train"].features["labels"].names
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}
print("Classes :", labels)
print("id2label :", id2label)

## 4. Préprocesseur d'images (Image Processor)

L'`AutoImageProcessor` connaît la taille d'entrée et la normalisation attendues par le ViT.

In [ ]:
from transformers import AutoImageProcessor

checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
print(image_processor)

## 5. Transformations (data augmentation)

On applique un recadrage aléatoire à l'entraînement (augmentation) et un redimensionnement simple à la validation. On remplace la colonne `image` par des tenseurs `pixel_values`.

In [ ]:
from torchvision.transforms import (
    Compose, Normalize, ToTensor, RandomResizedCrop, Resize, CenterCrop
)

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
crop = size if isinstance(size, int) else size[0]

train_tf = Compose([RandomResizedCrop(crop), ToTensor(), normalize])
val_tf   = Compose([Resize(crop), CenterCrop(crop), ToTensor(), normalize])

def apply_train(batch):
    batch["pixel_values"] = [train_tf(img.convert("RGB")) for img in batch["image"]]
    del batch["image"]
    return batch

def apply_val(batch):
    batch["pixel_values"] = [val_tf(img.convert("RGB")) for img in batch["image"]]
    del batch["image"]
    return batch

dataset["train"].set_transform(apply_train)
dataset["validation"].set_transform(apply_val)
dataset["test"].set_transform(apply_val)
print("Transformations appliquées.")

## 6. Collator et métrique d'évaluation

In [ ]:
import numpy as np
import evaluate
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

## 7. Chargement du modèle

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

## 8. Configuration de l'entraînement

Remplacez `vit-beans-demo` par le nom de dépôt que vous voulez. Avec `push_to_hub=True`,
le modèle sera publié sur `https://huggingface.co/VOTRE_NOM/vit-beans-demo`.

> Notes API (versions récentes de `transformers`) : le paramètre s'appelle **`eval_strategy`** (et non `evaluation_strategy`), et le `Trainer` reçoit **`processing_class`** (et non `tokenizer`).

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="vit-beans-demo",
    remove_unused_columns=False,   # IMPORTANT : garde 'image' pour créer pixel_values
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",              # pas de logger externe pour l'atelier
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=image_processor,
    compute_metrics=compute_metrics,
)

## 9. Entraînement

In [ ]:
trainer.train()

## 10. Évaluation sur le jeu de test

In [ ]:
metrics = trainer.evaluate(dataset["test"])
print(metrics)

## 11. Publication du modèle sur le Hub

`push_to_hub()` envoie les poids, la configuration, le processeur d'images et une *model card* générée automatiquement.

In [ ]:
trainer.push_to_hub()
print("Modèle publié ! Retrouvez-le sur votre profil Hugging Face.")

## 12. Test rapide de l'inférence

On recharge le modèle publié via un `pipeline` et on le teste sur une image du jeu de test.

In [ ]:
from transformers import pipeline
from huggingface_hub import whoami

user = whoami()["name"]
repo_id = f"{user}/vit-beans-demo"
print("Modèle :", repo_id)

classifier = pipeline("image-classification", model=repo_id)

# On recharge une image brute (sans transform) pour le test
raw = load_dataset("beans", split="test")
img = raw[0]["image"]
print("Vraie classe :", id2label[raw[0]["labels"]])
print("Prédictions :", classifier(img))
img

## ✅ Étape suivante : créer un Space Gradio

Votre modèle est en ligne. Passez maintenant au guide du lab (section **Space Gradio**) pour construire une interface web :
1. Créez un nouveau **Space** (SDK : *Gradio*) sur Hugging Face.
2. Ajoutez `app.py` et `requirements.txt` (fournis dans le dossier `space/`).
3. Remplacez `YOUR_USERNAME/vit-beans-demo` par votre `repo_id`.
4. Le Space se construit et sert une interface où l'on dépose une image de feuille pour obtenir la classe prédite.
